### Eupatilin - Protein Docking 을 위한 데이터 준비 ###
* Eupatilin smiles : O=C1C=C(C2=CC=C(OC)C(OC)=C2)OC3=CC(O)=C(OC)C(O)=C13
* Protein PDB : 엑셀 파일 참고

In [1]:
import subprocess
from pathlib import Path

from rdkit import Chem
from rdkit.Chem import rdMolDescriptors, Draw, AllChem, Descriptors, SDWriter
from rdkit.Chem.Draw import rdMolDraw2D

import warnings
import subprocess
from pathlib import Path

import nglview as nv
from openbabel import pybel
from opencadd.structure.core import Structure
import MDAnalysis as mda

import pandas as pd
import os

In [2]:
os.getcwd()

'/home/jeongin/eupatilin/preprocessing'

In [3]:
#문자 > 아스키 코드로 변환
# ascii code 65 = "A"
# ord() / chr()
a = ord("A")
a

65

In [ ]:
smiles = "O=C1C=C(C2=CC=C(OC)C(OC)=C2)OC3=CC(O)=C(OC)C(O)=C13"
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol) # 수소 추가

# ETKDGv3 3D 좌표 생성 임베딩
AllChem.EmbedMolecule(mol, AllChem.ETKDG())

# MMFF94 최적화 / 에너지 평가
AllChem.MMFFOptimizeMolecule(mol)

# 최저 에너지 컨포머만 저장
w = SDWriter("1_eupatilin_3D.sdf")
w.write(mol)
w.close()

In [5]:
#CC(Cc1ccc2c(c1)OCO2)C(C)Cc3ccc4c(c3)OCO4

smiles = "CC(Cc1ccc2c(c1)OCO2)C(C)Cc3ccc4c(c3)OCO4"
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol) # 수소 추가

# ETKDGv3 3D 좌표 생성 임베딩
AllChem.EmbedMolecule(mol, AllChem.ETKDG())

# MMFF94 최적화 / 에너지 평가
AllChem.MMFFOptimizeMolecule(mol)

# 최저 에너지 컨포머만 저장
w = SDWriter("9G9.sdf")
w.write(mol)
w.close()

In [ ]:
# conda = meeko
mk_prepare_ligand.py --read_pdb /home/jeongin/eupatilin/data/KMU-11421/8TB5.pdb -o 9G9.pdbqt

In [ ]:
# pdbqt > pdb (정확한 3D 구조 확인용)
obabel /home/jeongin/eupatilin/eupatilin.pdbqt -O /home/jeongin/eupatilin/eupatilin.pdb

# sdf (smiles) > pdb (3D 구조 확인용)
obabel -:"O=C1C=C(C2=CC=C(OC)C(OC)=C2)OC3=CC(O)=C(OC)C(O)=C13" --gen3d -O /home/jeongin/eupatilin/eupatilin_original.pdb

### Protein cif > pdb > pdbqt ###

In [3]:
import warnings
import subprocess
from pathlib import Path

import nglview as nv
from openbabel import pybel
from opencadd.structure.core import Structure
import MDAnalysis as mda

import pandas as pd

In [2]:
# cif to pdb
from pathlib import Path
from typing import Union
import gemmi

def cif_to_pdb(cif_path: Union[str, Path], pdb_path: Union[str, Path]) -> None:
    cif_path = Path(cif_path)
    pdb_path = Path(pdb_path)
    pdb_path.parent.mkdir(parents=True, exist_ok=True)

    doc = gemmi.cif.read_file(str(cif_path))
    block = doc.sole_block()
    structure = gemmi.make_structure_from_block(block)
    structure.write_pdb(str(pdb_path))
    print(f"✅ {cif_path} → {pdb_path}")

def convert_recursive(root: Union[str, Path], out_subdir: str = "pdb", suffix: str = ".pdb") -> None:
    root = Path(root).resolve()
    for cif in sorted(root.rglob("*.cif")):
        out_dir = cif.parent / out_subdir
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / (cif.stem + suffix)
        try:
            cif_to_pdb(cif, out_path)
        except Exception as e:
            print(f"❌ 변환 실패: {cif} ({e})")

convert_recursive("/home/jeongin/eupatilin/data/RCAN1")

✅ /home/jeongin/eupatilin/data/RCAN1/6UUQ.cif → /home/jeongin/eupatilin/data/RCAN1/pdb/6UUQ.pdb


In [1]:
# cif to pdb
from pathlib import Path
from typing import Union
import gemmi

def cif_to_pdb(cif_path: Union[str, Path], pdb_path: Union[str, Path]) -> None:
    cif_path = Path(cif_path)
    pdb_path = Path(pdb_path)
    pdb_path.parent.mkdir(parents=True, exist_ok=True)

    doc = gemmi.cif.read_file(str(cif_path))
    block = doc.sole_block()
    structure = gemmi.make_structure_from_block(block)
    structure.write_pdb(str(pdb_path))
    print(f"✅ {cif_path} → {pdb_path}")

def convert_once_level(root: Union[str, Path], out_subdir: str = "pdb", suffix: str = ".pdb") -> None:
    root = Path(root).resolve()
    for sub in sorted(root.iterdir()):
        if not sub.is_dir():
            continue  # 파일 스킵
        cif_files = sorted(sub.glob("*.cif"))  
        if not cif_files:
            continue

        out_dir = sub / out_subdir
        for cif in cif_files:
            out_path = out_dir / (cif.stem + suffix)
            try:
                cif_to_pdb(cif, out_path)
            except Exception as e:
                print(f"❌ 변환 실패: {cif} ({e})")

convert_once_level("/home/jeongin/eupatilin/data/negative_protein")


✅ /home/jeongin/eupatilin/data/negative_protein/GR/3K22.cif → /home/jeongin/eupatilin/data/negative_protein/GR/pdb/3K22.pdb
✅ /home/jeongin/eupatilin/data/negative_protein/GR/4LSJ.cif → /home/jeongin/eupatilin/data/negative_protein/GR/pdb/4LSJ.pdb
✅ /home/jeongin/eupatilin/data/negative_protein/GR/4MDD.cif → /home/jeongin/eupatilin/data/negative_protein/GR/pdb/4MDD.pdb
✅ /home/jeongin/eupatilin/data/negative_protein/GR/4UDC.cif → /home/jeongin/eupatilin/data/negative_protein/GR/pdb/4UDC.pdb
✅ /home/jeongin/eupatilin/data/negative_protein/GR/6DXK.cif → /home/jeongin/eupatilin/data/negative_protein/GR/pdb/6DXK.pdb
✅ /home/jeongin/eupatilin/data/negative_protein/GR/8VKZ.cif → /home/jeongin/eupatilin/data/negative_protein/GR/pdb/8VKZ.pdb
✅ /home/jeongin/eupatilin/data/negative_protein/HK1/1hkc.cif → /home/jeongin/eupatilin/data/negative_protein/HK1/pdb/1hkc.pdb
✅ /home/jeongin/eupatilin/data/negative_protein/HK1/4F9O.cif → /home/jeongin/eupatilin/data/negative_protein/HK1/pdb/4F9O.pdb
✅ /h

In [2]:
protein_pdbs = sorted(Path("/home/jeongin/eupatilin/data/negative_protein").rglob("pdb/*.pdb"))
protein_pdbs

[PosixPath('/home/jeongin/eupatilin/data/negative_protein/GR/pdb/3K22.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/GR/pdb/4LSJ.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/GR/pdb/4MDD.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/GR/pdb/4UDC.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/GR/pdb/6DXK.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/GR/pdb/8VKZ.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/HK1/pdb/1hkc.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/HK1/pdb/4F9O.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/HK1/pdb/4FPB.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/HK2/pdb/5HEX.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/HK2/pdb/5HG1.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/PPAR_gamma/pdb/2ZNO.pdb'),
 PosixPath('/home/jeongin/eupatilin/data/negative_protein/PPAR_

In [3]:
# pdb > csv

from pathlib import Path
from typing import Union, Optional, Tuple, Dict
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import cpu_count
from Bio.PDB import PDBParser
import pandas as pd
import traceback
import sys

# ---------- 단일 PDB → CSV ----------
def _infer_element(atom_name: str) -> str:
    name = atom_name.strip()
    if not name:
        return ""
    if name[0].isdigit():
        cand = name[1:3]
    else:
        cand = name[:2]
    return cand.capitalize().strip()

def pdb_to_csv_single(pdb_path: Union[str, Path], csv_path: Union[str, Path]) -> Tuple[str, bool, Optional[str], int]:
    """
    Returns: (pdb_path, success, error_message, atom_count)
    """
    pdb_path = Path(pdb_path)
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_path.stem.upper(), str(pdb_path))

    rows = []
    atom_id = 1

    for model in structure:
        model_id = getattr(model, "id", 0)
        for chain in model:
            chain_id = chain.id
            for residue in chain:
                res_id = residue.get_id()
                res_num = res_id[1] if isinstance(res_id, tuple) else res_id
                icode = res_id[2] if isinstance(res_id, tuple) and len(res_id) > 2 else ""
                resname = residue.get_resname().strip()

                for atom in residue:
                    coords = atom.get_coord()
                    elem = getattr(atom, "element", "") or _infer_element(atom.get_name())
                    rows.append({
                        "id": atom_id,
                        "pdb_code": pdb_path.stem.upper(),
                        "model_id": model_id,
                        "chain": chain_id,
                        "residue_name": resname,
                        "residue_number": int(res_num) if isinstance(res_num, int) else res_num,
                        "insertion_code": icode.strip(),
                        "atom_name": atom.get_name().strip(),
                        "x": float(coords[0]),
                        "y": float(coords[1]),
                        "z": float(coords[2]),
                        "occupancy": float(atom.get_occupancy() or 0.0),
                        "bfactor": float(atom.get_bfactor() or 0.0),
                        "element": elem
                    })
                    atom_id += 1

    if not rows:
        return (str(pdb_path), False, "no atoms parsed", 0)

    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    return (str(pdb_path), True, None, len(df))

# ---------- 배치 실행 ----------
def batch_pdb_to_csv(
    root: Union[str, Path],
    out_root: Optional[Union[str, Path]] = None,
    overwrite: bool = False,
    max_workers: Optional[int] = None,
) -> Dict[str, str]:
    """
    root 아래 모든 *.pdb 변환.
    - out_root=None: PDB 파일과 같은 폴더에 <파일명>.csv 저장
    - out_root=경로: root 기준 상대경로를 보존하여 out_root 밑에 미러링
    Returns: 결과 요약 dict
    """
    root = Path(root).resolve()
    out_root = Path(out_root).resolve() if out_root else None
    max_workers = max_workers or max(1, cpu_count() - 1)

    pdb_files = sorted(root.rglob("*.pdb"))
    if not pdb_files:
        print("⚠️ No PDB files found.")
        return {"found": "0", "converted": "0", "skipped": "0", "failed": "0"}

    tasks = []
    converted = skipped = failed = 0

    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = {}
        for pdb in pdb_files:
            if out_root:
                rel = pdb.relative_to(root)
                csv_path = (out_root / rel).with_suffix(".csv")
            else:
                csv_path = pdb.with_suffix(".csv")

            if csv_path.exists() and not overwrite:
                skipped += 1
                continue

            futures[ex.submit(pdb_to_csv_single, pdb, csv_path)] = (pdb, csv_path)

        for fut in as_completed(futures):
            pdb, csv_path = futures[fut]
            try:
                pdb_str, ok, err, n_atoms = fut.result()
                if ok:
                    converted += 1
                    print(f"✅ {pdb.name} → {csv_path} ({n_atoms} atoms)")
                else:
                    failed += 1
                    print(f"❌ {pdb.name}: {err}")
            except Exception:
                failed += 1
                print(f"❌ {pdb.name}: {traceback.format_exc().splitlines()[-1]}")

    summary = {
        "found": str(len(pdb_files)),
        "converted": str(converted),
        "skipped": str(skipped),
        "failed": str(failed),
    }
    print(f"\n=== Summary ===\nFound: {summary['found']}, Converted: {summary['converted']}, Skipped: {summary['skipped']}, Failed: {summary['failed']}")
    return summary

if __name__ == "__main__":
   
    batch_pdb_to_csv(
        root="/home/jeongin/eupatilin/data/protein/PPAR_gamma/chain_split",
        out_root=None,  # 원본 옆에 두고 싶으면 None
        overwrite=False,                              # 이미 있으면 건너뜀
        max_workers=None                              # 자동(코어수-1)
    )


✅ 2ZNO.pdb → /home/jeongin/eupatilin/data/protein/PPAR_gamma/chain_split/2ZNO.csv (2172 atoms)
✅ 3KMG.pdb → /home/jeongin/eupatilin/data/protein/PPAR_gamma/chain_split/3KMG.csv (2251 atoms)
✅ 3NOA.pdb → /home/jeongin/eupatilin/data/protein/PPAR_gamma/chain_split/3NOA.csv (2254 atoms)
✅ 4EM9.pdb → /home/jeongin/eupatilin/data/protein/PPAR_gamma/chain_split/4EM9.csv (2255 atoms)

=== Summary ===
Found: 4, Converted: 4, Skipped: 0, Failed: 0


In [3]:
# pdb > csv

from pathlib import Path
from typing import Union, Optional, Tuple, Dict
from concurrent.futures import ProcessPoolExecutor, as_completed
from multiprocessing import cpu_count
from Bio.PDB import PDBParser
import pandas as pd
import traceback
import sys

# ---------- 단일 PDB → CSV ----------
def _infer_element(atom_name: str) -> str:
    name = atom_name.strip()
    if not name:
        return ""
    if name[0].isdigit():
        cand = name[1:3]
    else:
        cand = name[:2]
    return cand.capitalize().strip()

def pdb_to_csv_single(pdb_path: Union[str, Path], csv_path: Union[str, Path]) -> Tuple[str, bool, Optional[str], int]:
    """
    Returns: (pdb_path, success, error_message, atom_count)
    """
    pdb_path = Path(pdb_path)
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_path.stem.upper(), str(pdb_path))

    rows = []
    atom_id = 1

    for model in structure:
        model_id = getattr(model, "id", 0)
        for chain in model:
            chain_id = chain.id
            for residue in chain:
                res_id = residue.get_id()
                res_num = res_id[1] if isinstance(res_id, tuple) else res_id
                icode = res_id[2] if isinstance(res_id, tuple) and len(res_id) > 2 else ""
                resname = residue.get_resname().strip()

                for atom in residue:
                    coords = atom.get_coord()
                    elem = getattr(atom, "element", "") or _infer_element(atom.get_name())
                    rows.append({
                        "id": atom_id,
                        "pdb_code": pdb_path.stem.upper(),
                        "model_id": model_id,
                        "chain": chain_id,
                        "residue_name": resname,
                        "residue_number": int(res_num) if isinstance(res_num, int) else res_num,
                        "insertion_code": icode.strip(),
                        "atom_name": atom.get_name().strip(),
                        "x": float(coords[0]),
                        "y": float(coords[1]),
                        "z": float(coords[2]),
                        "occupancy": float(atom.get_occupancy() or 0.0),
                        "bfactor": float(atom.get_bfactor() or 0.0),
                        "element": elem
                    })
                    atom_id += 1

    if not rows:
        return (str(pdb_path), False, "no atoms parsed", 0)

    df = pd.DataFrame(rows)
    df.to_csv(csv_path, index=False)
    return (str(pdb_path), True, None, len(df))

# ---------- 배치 실행 ----------
def batch_pdb_to_csv(
    root: Union[str, Path],
    out_root: Optional[Union[str, Path]] = None,
    overwrite: bool = False,
    max_workers: Optional[int] = None,
) -> Dict[str, str]:
    """
    root 아래 모든 *.pdb 변환.
    - out_root=None: PDB 파일과 같은 폴더에 <파일명>.csv 저장
    - out_root=경로: root 기준 상대경로를 보존하여 out_root 밑에 미러링
    Returns: 결과 요약 dict
    """
    root = Path(root).resolve()
    out_root = Path(out_root).resolve() if out_root else None
    max_workers = max_workers or max(1, cpu_count() - 1)

    pdb_files = sorted(root.rglob("*.pdb"))
    if not pdb_files:
        print("⚠️ No PDB files found.")
        return {"found": "0", "converted": "0", "skipped": "0", "failed": "0"}

    tasks = []
    converted = skipped = failed = 0

    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = {}
        for pdb in pdb_files:
            if out_root:
                rel = pdb.relative_to(root)
                csv_path = (out_root / rel).with_suffix(".csv")
            else:
                csv_path = pdb.with_suffix(".csv")

            if csv_path.exists() and not overwrite:
                skipped += 1
                continue

            futures[ex.submit(pdb_to_csv_single, pdb, csv_path)] = (pdb, csv_path)

        for fut in as_completed(futures):
            pdb, csv_path = futures[fut]
            try:
                pdb_str, ok, err, n_atoms = fut.result()
                if ok:
                    converted += 1
                    print(f"✅ {pdb.name} → {csv_path} ({n_atoms} atoms)")
                else:
                    failed += 1
                    print(f"❌ {pdb.name}: {err}")
            except Exception:
                failed += 1
                print(f"❌ {pdb.name}: {traceback.format_exc().splitlines()[-1]}")

    summary = {
        "found": str(len(pdb_files)),
        "converted": str(converted),
        "skipped": str(skipped),
        "failed": str(failed),
    }
    print(f"\n=== Summary ===\nFound: {summary['found']}, Converted: {summary['converted']}, Skipped: {summary['skipped']}, Failed: {summary['failed']}")
    return summary

if __name__ == "__main__":
   
    batch_pdb_to_csv(
        root="/home/jeongin/eupatilin/data/negative_protein",
        out_root="/home/jeongin/eupatilin/data/negative_protein/csv_out",  # 원본 옆에 두고 싶으면 None
        overwrite=False,                              # 이미 있으면 건너뜀
        max_workers=None                              # 자동(코어수-1)
    )


✅ 2ZNO_ligand.pdb → /home/jeongin/eupatilin/data/negative_protein/csv_out/PPAR_gamma/split_ligand/2ZNO_ligand.csv (33 atoms)
✅ 4EM9_ligand.pdb → /home/jeongin/eupatilin/data/negative_protein/csv_out/PPAR_gamma/split_ligand/4EM9_ligand.csv (11 atoms)
✅ 3KMG_ligand.pdb → /home/jeongin/eupatilin/data/negative_protein/csv_out/PPAR_gamma/split_ligand/3KMG_ligand.csv (39 atoms)
✅ 5F9B_ligand.pdb → /home/jeongin/eupatilin/data/negative_protein/csv_out/PPAR_gamma/split_ligand/5F9B_ligand.csv (35 atoms)
✅ 3NOA_ligand.pdb → /home/jeongin/eupatilin/data/negative_protein/csv_out/PPAR_gamma/split_ligand/3NOA_ligand.csv (41 atoms)
✅ 3KMG_protein.pdb → /home/jeongin/eupatilin/data/negative_protein/csv_out/PPAR_gamma/split_protein/3KMG_protein.csv (2106 atoms)
✅ 4LSJ.pdb → /home/jeongin/eupatilin/data/negative_protein/csv_out/GR/pdb/4LSJ.csv (2177 atoms)
✅ 2ZNO.pdb → /home/jeongin/eupatilin/data/negative_protein/csv_out/PPAR_gamma/chain_split/2ZNO.csv (2172 atoms)
✅ 2ZNO_protein.pdb → /home/jeongin/eu

### Protein - ligand 분리 순서 ###
* protein 구조 파악 (ligand, ion, small molecule, peptide linking 결합 확인,  chain 분리 필요한지 확인)
* ligand 기능을 하는 chain 만 남겨두고 단일 chain으로 분리 
* mmCIF + gemmi 사용 ion, small molecule, peptide linking 분리하여 protein-ligand 파일 생성 
* 부가 화합물 포함 구조는 분리 후 제대로 분리되었는지 원래 구조와 비교, 확인 필요함
* 일부 구조에서 LINK 누락 되는 경우 있음 > linking이 있는 protein은 모두 직접 확인 필요함
* LINK 포함 구조인데 PDB 파일 내에서 LINK 정보가 누락 > pymol에서 직접 원자 위치 select 해서 제거하여 pdb 파일 생성 
* Glycerol, methyl 같은 부가 화합물도 어떤 residue와 interaction 하고 있는지 위치 직접 확인 후 PDB 파일 구조내에서 정보 확인 필요함 

In [ ]:
# protein - ligand 분리
"""ion, peptide linking, ligand 결합의 경우 mmCIF 사용해 분리 권장"""
import os
from pathlib import Path
from openbabel import pybel

# 당장 docking 가능한 protein list > ion, peptide linking, ligand 구조인 경우 protein-ligand 분리 가능 , + 그 외 결합은 확인 필요(소분자, glyserol 등)
""" Positive protein list 
- ERK1 = 2ZOQ (ion 2, peptide linking, ligand)
- ERK1 = 4QTB (ion 2, ligand, small molecule)
- ERK2 = 6SLG (ligand)
- ERK2 = 4FV1 (ion1, glyserol, peptide linking, ligand)
- ERK2 = 4FV6 (ion, ethanediol, ion, ligand)
- p38 MARK = 6SFO (ion, glyserol, ligand)
- JNK2 = 3E7O (ligand) / dimer 형태
- JNK2 = 7N8T (glycol, ligand) 
- JNK2 = 3NPC (ligand) / dimer 형태
- GSK = 5HLN (ion, peptide linking, ligand) / dimer 형태
- GSK = 6GN1 (ion2, ligand) / dimer 형태
- GSK = 3I4B (ligand) / dimer 형태
- GSK = 4J1R (ion2, peptide linking, ligand) / dimer 형태
- GSK = 4JI7 (ion2, peptide linking, small ligand) / dimer 형태
- GSK = 3SAY (acid2,methyl pentanediol, peptide linking, ligand) / dimer 형태
- GSK = 5F94 (ligand) / dimer 형태
- GSK = 5F95 (ligand) / dimer 형태
- AKT1 = 3O96 (MolWt 큰 ligand) / 
- AKT1 = 4EKL (peptide linking, ligand) / 
- AKT1 = 8UW7 (ion, ethanediol, piptide linking, ligand) + B chain 확인 필요
- AKT1 = 3MV5 (ion, peptide linking, ligand) + B chain이 긴 체인 형태 - 확인 필요 
- PPAR alpha = 3VI8 (ligand)
- PPAR alpha = 6KXY (ligand)
- PPAR alpha  = 8YT9 (ligand)
- PPAR gamma = 2ZNO (ligand) / dimer 형태
- PPAR gamma = 3KMG (ligand) / dimer 형태, B 체인 남기기 (alpha 체인으로 붙어있음)
- PPAR gamma = 3NOA (ligand) / dimer 형태
- PPAR gamma = 4EM9 (알킬체인) / dimer 형태
- PPAR gamma = 4YT1 (ligand) / dimer X, reference ligand interaction 이 없음
- PPAR gamma = 5F9B (ligand) / dimer X, ligand 결합 부위는 한 군데만 있음 
"""

""" Negative protein list
- GR = 3K22 (ligand) / dimer 형태 + B chain
- GR = 4LSJ (ligand) + B chain
- GR = 4MDD (ligand) / dimer 형태 + C, D chain = nuclear receptor
- GR = 4UDC (ligand + DEXAMETHASONE) / dimer 형태 + B chain(nulear receptor) - DEXAMETHASONE을 ligand로 분리할 것 
- GR = 6DXK (compound) / dimer 형태 
- HK1 = 1HKC (glucopyranose, ion) - docking 불가
- HK1 = 4F9O (2-deoxy...beta..glucopyranose) / dimer 형태 / 구조 내 포타슘 이온 있음
- HK1 = 4FPB (1,5-anhydro-D-glucopyranose) / dimer 형태
- HK2 = 5HEX (compound 6 benzo..glucosamine) / dimer 형태
- HK2 = 5HG1 (compound C-2 glucosamine) / dimer 형태
- TrkA = 4PMP (inhibitor) 
- TrkA = 5H3Q (inhibitor) / 구조 내 포타슘 이온 있음
- TrkA = 5JFS (inhibitor)
- TrkA = 8J5W (inhibitor)

"""


In [2]:
# 직접 분리

import gemmi
from pathlib import Path

# ===== 기본 설정 =====
pdb_file = Path("/home/jeongin/eupatilin/data/KMU-11421/8TB5.pdb")

standard_aas = {
    "ALA", "ARG", "ASN", "ASP", "CYS",
    "GLU", "GLN", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO",
    "SER", "THR", "TRP", "TYR", "VAL"
}

modified_residues = {
    "MSE", "TPO", "SEP", "PTR", "YCM", "CSO", "CME", "MLY",
    "CSX", "CSD", "HIP", "HID", "HIE", "HYP"
}

# 완전히 제거할 것들
solvent_like = {"HOH", "WAT", "DOD", "SOL", "TIP3"}

ion_like = {
    "NA", "K", "CL", "BR", "I",
    "CA", "MG", "MN", "FE", "FE2", "ZN", "CU", "CO",
    "NI", "CD", "HG", "SR", "CS", "LI", "BA", "YB"
}

buffer_like = {
    "HEP", "EPE", "TRS", "MES", "PIP", "MPO", "ADA",
    "GOL", "PEG", "PG4", "PG5", "PG6",
    "ACT", "ACE", "CIT", "FMT", "SO4", "PO4",
    "EOH", "IPA", "DMS", "EDO", "TCE", "PGE"
}

exclude_residues = solvent_like | ion_like | buffer_like


# ===== PDB 파일 처리 =====
try:
    print(f"[INFO] 처리 중: {pdb_file}")

    st = gemmi.read_structure(str(pdb_file))

    prot_st = gemmi.Structure()
    lig_st = gemmi.Structure()
    prot_st.name = st.name
    lig_st.name = st.name

    parsed_residues = set()

    # --- ① Gemmi 계층 기반 탐색 ---
    for model in st:
        prot_model = gemmi.Model(model.name)
        lig_model = gemmi.Model(model.name)
        for chain in model:
            chain_name = chain.name.strip() if chain.name.strip() else "_"
            prot_chain = gemmi.Chain(chain_name)
            lig_chain = gemmi.Chain(chain_name)
            for res in chain:
                resname = res.name.strip().upper()
                parsed_residues.add(resname)
                if resname in standard_aas or resname in modified_residues:
                    prot_chain.add_residue(res)
                elif resname not in exclude_residues:
                    lig_chain.add_residue(res)
            if len(prot_chain):
                prot_model.add_chain(prot_chain)
            if len(lig_chain):
                lig_model.add_chain(lig_chain)
        if len(prot_model):
            prot_st.add_model(prot_model)
        if len(lig_model):
            lig_st.add_model(lig_model)

    # --- ② line-based 보충 탐색 ---
    unparsed_lines = []
    with open(pdb_file) as f:
        for line in f:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            resname = line[17:20].strip().upper()
            if resname not in parsed_residues:
                unparsed_lines.append(line)

    if unparsed_lines:
        print(f"[WARN] 계층 구조에서 누락된 원자 {len(unparsed_lines)}개 → line 기반으로 보완")
        for line in unparsed_lines:
            resname = line[17:20].strip().upper()
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
            atom_name = line[12:16].strip()

            if resname in standard_aas or resname in modified_residues:
                model = prot_st[0] if len(prot_st) > 0 else prot_st.add_model("0")
                chain = model.find_chain("_") or model.add_chain("_")
                residue = gemmi.make_residue("UNK", 0, ' ')
                residue.name = resname
                atom = gemmi.make_small_atom(atom_name, x, y, z)
                residue.add_atom(atom)
                chain.add_residue(residue)

            elif resname not in exclude_residues:
                model = lig_st[0] if len(lig_st) > 0 else lig_st.add_model("0")
                chain = model.find_chain("_") or model.add_chain("_")
                residue = gemmi.make_residue("UNK", 0, ' ')
                residue.name = resname
                atom = gemmi.make_small_atom(atom_name, x, y, z)
                residue.add_atom(atom)
                chain.add_residue(residue)

    # --- ③ 결과 저장 ---
    protein_out = pdb_file.parent / f"{pdb_file.stem}_protein.pdb"
    ligand_out  = pdb_file.parent / f"{pdb_file.stem}_ligand.pdb"

    if len(prot_st) > 0:
        prot_st.write_minimal_pdb(str(protein_out))
        print(f"✅ 단백질 저장 완료: {protein_out}")
    if len(lig_st) > 0:
        lig_st.write_minimal_pdb(str(ligand_out))
        print(f"✅ 리간드 저장 완료: {ligand_out}")

    print()

except Exception as e:
    print(f"❌ 오류 발생 ({pdb_file.name}): {e}")

[INFO] 처리 중: /home/jeongin/eupatilin/data/KMU-11421/8TB5.pdb
✅ 단백질 저장 완료: /home/jeongin/eupatilin/data/KMU-11421/8TB5_protein.pdb
✅ 리간드 저장 완료: /home/jeongin/eupatilin/data/KMU-11421/8TB5_ligand.pdb



# Protein pdb > pdbqt
* Meeko 사용

In [ ]:
/home/koxpular/miniconda3/envs/meeko/bin/mk_prepare_receptor.py \
  --read_pdb 8TB5_protein.pdb \
  -o 8TB5_protein_meeko \
  -p -a --default_altloc A

# Ligand sdf > pdbqt
* Meeko 사용

In [ ]:
/home/koxpular/miniconda3/envs/meeko/bin/mk_prepare_ligand.py -i 8tb5_F_ZOQ.sdf -o ZOQ.pdbqt

### Docking simulation ###
* docking 한 번에 여러개 돌릴 수 있는 코드로 수정하기 
* csv 파일에서 읽어오던지 ..  아무튼 여러개 한 번에 도킹 할 수 있는 코드로 수정 필요함
* docking_simulation.py
* 한 번에 돌리니까 오류나거나 못 읽는 파일도 있어서 그냥 하나씩 돌리기로함

In [4]:
#!/usr/bin/env python3
import subprocess, os, ast
from pathlib import Path
import pandas as pd

SMINA_BIN = "/home/jeongin/eupatilin/smina.linux"
COMPOUND  = Path("/home/jeongin/eupatilin/data/KMU-11421/split_pdb/ZOQ.pdbqt")
PROTEIN   = Path("/home/jeongin/eupatilin/data/KMU-11421/split_pdb/8TB5_protein_meeko.pdbqt")
OUT_DIR   = Path("/home/jeongin/eupatilin/data/KMU-11421/docking_results"); OUT_DIR.mkdir(parents=True, exist_ok=True)
POCKET_CSV= Path("/home/jeongin/eupatilin/data/KMU-11421/pocket_info/8TB5_pocket_info.csv")

def parse_vec(v):
    if isinstance(v, str):
        v = ast.literal_eval(v)  # 예: "[10.0, 20.0, 30.0]"
    return tuple(float(x) for x in v)

def load_center_size_rows(csv_path: Path):
    """두 포맷 모두 지원:
       A) center_x,center_y,center_z,size_x,size_y,size_z
       B) 'pocket center','pocket size' (각 셀에 [x,y,z] 문자열)
       반환: [(center_tuple, size_tuple), ...]
    """
    df = pd.read_csv(csv_path)
    rows = []
    if {"center_x","center_y","center_z","size_x","size_y","size_z"}.issubset(df.columns):
        for _, r in df.iterrows():
            c = (float(r["center_x"]), float(r["center_y"]), float(r["center_z"]))
            s = (float(r["size_x"]),   float(r["size_y"]),   float(r["size_z"]))
            rows.append((c, s))
    elif {"pocket center","pocket size"}.issubset(df.columns):
        for _, r in df.iterrows():
            c = parse_vec(r["pocket center"])
            s = parse_vec(r["pocket size"])
            rows.append((c, s))
    else:
        raise ValueError("CSV에 필요한 컬럼이 없습니다. (center_x..size_z) 또는 ('pocket center','pocket size')")
    return rows

def run_smina(ligand_path, protein_path, out_path, pocket_center, pocket_size,
              num_poses=10, exhaustiveness=8, energy_range=3, log_path=None):
    out_path = Path(out_path)
    if out_path.suffix.lower() != ".pdbqt":
        out_path = out_path.with_suffix(".pdbqt")
    cx, cy, cz = map(float, pocket_center)
    sx, sy, sz = map(float, pocket_size)
    cmd = [
        SMINA_BIN,
        "--ligand", str(ligand_path),
        "--receptor", str(protein_path),
        "--out", str(out_path),
        "--center_x", str(cx), "--center_y", str(cy), "--center_z", str(cz),
        "--size_x", str(sx),  "--size_y", str(sy),  "--size_z", str(sz),
        "--num_modes", str(num_poses),
        "--energy_range", str(energy_range),
        "--exhaustiveness", str(exhaustiveness)
    ]
    if log_path:
        cmd += ["--log", str(log_path)]
    # smina 표를 캡처해서 반환(원하면 실시간 출력으로 바꿀 수 있음)
    output_text = subprocess.check_output(cmd, text=True)
    return output_text, out_path

def parse_best_from_pose(pose_path: Path):
    """PDBQT 내부 'REMARK  VINA RESULT:'에서 최저(가장 좋은) 스코어 추출"""
    scores = []
    for line in Path(pose_path).read_text(errors="ignore").splitlines():
        if "REMARK" in line and "VINA RESULT" in line:
            parts = line.strip().split()
            for tok in parts:
                try:
                    scores.append(float(tok)); break
                except:
                    pass
    return min(scores) if scores else None, len(scores)

def main():
    boxes = load_center_size_rows(POCKET_CSV)

    if len(boxes) == 1:
        center, size = boxes[0]
        out_pose = OUT_DIR / "8TB5_docked.pdbqt"
        out_log  = OUT_DIR / "8TB5.log"
        stdout_txt, pose_path = run_smina(COMPOUND, PROTEIN, out_pose, center, size,
                                          num_poses=10, exhaustiveness=8, energy_range=3,
                                          log_path=out_log)
        print(stdout_txt)
        best, nposes = parse_best_from_pose(pose_path)
        print(f"[SUMMARY] poses={nposes}  best_affinity={best} kcal/mol  -> {pose_path}")
    else:
        # 여러 행: 각 포켓에 대해 개별 파일로 도킹 (8TB5_p0_docked.pdbqt, ...)
        for i, (center, size) in enumerate(boxes):
            out_pose = OUT_DIR / f"8TB5_p{i}_docked.pdbqt"
            out_log  = OUT_DIR / f"8TB5_p{i}.log"
            stdout_txt, pose_path = run_smina(COMPOUND, PROTEIN, out_pose, center, size,
                                              num_poses=10, exhaustiveness=8, energy_range=3,
                                              log_path=out_log)
            print(stdout_txt)
            best, nposes = parse_best_from_pose(pose_path)
            print(f"[SUMMARY][p{i}] poses={nposes}  best_affinity={best} kcal/mol  -> {pose_path}")

if __name__ == "__main__":
    main()


   _______  _______ _________ _        _______ 
  (  ____ \(       )\__   __/( (    /|(  ___  )
  | (    \/| () () |   ) (   |  \  ( || (   ) |
  | (_____ | || || |   | |   |   \ | || (___) |
  (_____  )| |(_)| |   | |   | (\ \) ||  ___  |
        ) || |   | |   | |   | | \   || (   ) |
  /\____) || )   ( |___) (___| )  \  || )   ( |
  \_______)|/     \|\_______/|/    )_)|/     \|


smina is based off AutoDock Vina. Please cite appropriately.

Weights      Terms
-0.035579    gauss(o=0,_w=0.5,_c=8)
-0.005156    gauss(o=3,_w=2,_c=8)
0.840245     repulsion(o=0,_c=8)
-0.035069    hydrophobic(g=0.5,_b=1.5,_c=8)
-0.587439    non_dir_h_bond(g=-0.7,_b=0,_c=8)
1.923        num_tors_div

Using random seed: 467091610

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
1

In [12]:
# remove PO4-like residues from receptor PDB only (post-process)
from pathlib import Path

# 이전 분리 셀에서 REC_PDB가 있으면 재사용, 없으면 기본 경로 사용
rec_pdb_path = Path(str(REC_PDB)) if "REC_PDB" in globals() else Path("/home/jeongin/eupatilin/data/RCAN1/split/6UUQ_protein.pdb")
out_pdb_path = rec_pdb_path.with_name(rec_pdb_path.stem + "_no_po4.pdb")

PHOSPHATE_RESN = {"PO4", "HPO", "HPO4", "PI", "PHO"}

kept_lines = []
removed_count = 0
removed_residues = set()

with rec_pdb_path.open("r", errors="ignore") as f:
    for line in f:
        rec = line[0:6].strip()
        if rec in ("ATOM", "HETATM"):
            resn = line[17:20].strip().upper()
            chain = line[21].strip()
            try:
                resi = int(line[22:26])
            except ValueError:
                resi = line[22:26].strip()

            if resn in PHOSPHATE_RESN:
                removed_count += 1
                removed_residues.add((resn, chain, resi))
                continue
        kept_lines.append(line)

if not kept_lines or kept_lines[-1].strip() != "END":
    kept_lines.append("END\n")

out_pdb_path.write_text("".join(kept_lines))

print("✅ PO4 계열 제거 완료")
print(" - input :", rec_pdb_path)
print(" - output:", out_pdb_path)
print(" - removed atoms:", removed_count)
print(" - removed residues:", sorted(removed_residues))

✅ PO4 계열 제거 완료
 - input : /home/jeongin/eupatilin/data/RCAN1/split/6UUQ_protein.pdb
 - output: /home/jeongin/eupatilin/data/RCAN1/split/6UUQ_protein_no_po4.pdb
 - removed atoms: 5
 - removed residues: [('PO4', 'A', 401)]
